In [1]:
!pip install -q sentence-transformers faiss-cpu transformers pdfplumber streamlit openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.4 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import pathlib
import urllib.request
from glob import glob
from typing import List, Dict, Any


import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pdfplumber


# HF tokenizer/model (for optional use)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Directories
PDF_DIR = "download_pdfs"
DATA_DIR = "agri_corpus"
INDEX_DIR = "index_data"
pathlib.Path(PDF_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)


print("Setup complete. PDF_DIR:", PDF_DIR, "DATA_DIR:", DATA_DIR)

Setup complete. PDF_DIR: download_pdfs DATA_DIR: agri_corpus


In [3]:
pdf_urls = [
# FAO example (openknowledge) - replace if link expired
"https://openknowledge.fao.org/server/api/core/bitstreams/80f293c6-0b25-402b-b232-8fad0af6fc14/content",
# IRRI control of rice insect pests (example link)
"https://www.knowledgebank.irri.org/ericeproduction/PDF_%26_Docs/Control_of_rice_insect_pests.pdf",
# Example ICAR/NRRI bulletin (may change over time)
"https://icar-nrri.in/wp-content/uploads/2024/08/NRRI_Research-Bulletin-No-53.pdf",
# Example KVK/public extension note (may change)
"https://kvk.icar.gov.in/wp-content/uploads/2020/12/rice_ipm_notes.pdf"
]


print("URLs listed. You can add or remove URLs in pdf_urls list.")

URLs listed. You can add or remove URLs in pdf_urls list.


In [7]:
for url in pdf_urls:
    try:
        fname = os.path.join(PDF_DIR, os.path.basename(url.split("/")[-1]) or "doc.pdf")
# ensure unique filename
        if os.path.exists(fname):
            base, ext = os.path.splitext(fname)
            i = 1
            while os.path.exists(f"{base}_{i}{ext}"):
                i += 1
            fname = f"{base}_{i}{ext}"
        print("Downloading", url, "->", fname)
        urllib.request.urlretrieve(url, fname)
    except Exception as e:
        print("Failed to download", url, ":", e)


print("Download step done. Check the folder:", PDF_DIR)

Failed to download https://openknowledge.fao.org/server/api/core/bitstreams/80f293c6-0b25-402b-b232-8fad0af6fc14/content : HTTP Error 403: Forbidden
Failed to download https://www.knowledgebank.irri.org/ericeproduction/PDF_%26_Docs/Control_of_rice_insect_pests.pdf : <urlopen error [Errno 110] Connection timed out>
Failed to download https://kvk.icar.gov.in/wp-content/uploads/2020/12/rice_ipm_notes.pdf : <urlopen error [Errno -2] Name or service not known>
Download step done. Check the folder: download_pdfs


In [22]:
PEST_KEYWORDS = {
    "brown planthopper": ["planthopper", "nilaparvata", "bph", "brown planthopper"],
    "stem borer": ["stem borer", "scirpophaga", "borer", "stem-borer"],
    "leaf folder": ["leaf folder", "leaffolder", "leaf-folder"],
    "sheath blight": ["sheath blight", "rhizoctonia", "sheath-blight"],
    "general ipm": ["integrated pest management", "ipm", "monitor", "threshold"]
}

def tag_pest(text: str) -> List[str]:
    t = text.lower()
    tags = set()
    for pest, kws in PEST_KEYWORDS.items():
        for kw in kws:
            if kw in t:
                tags.add(pest)
                break
    return list(tags) if tags else ["unknown"]

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def chunk_text(text: str, chunk_words: int = 180, overlap: int = 40) -> List[str]:
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_words]
        chunks.append(" ".join(chunk))
        i += chunk_words - overlap
    return chunks

# Extract pages from downloaded PDFs
count = 0
pdf_files = sorted(list(pathlib.Path(PDF_DIR).glob("*.pdf")))
if not pdf_files:
    print("No PDFs found in", PDF_DIR, "- if you uploaded PDFs manually, ensure they are in the working directory or update PDF_DIR.")
else:
    for p in pdf_files:
        try:
            with pdfplumber.open(p) as pdf:
                for i, page in enumerate(pdf.pages):
                    text = page.extract_text() or ""
                    txt = clean_text(text)
                    if not txt:
                        continue
                    tags = tag_pest(txt)
                    chunks = chunk_text(txt, chunk_words=180, overlap=40)
                    for idx, ch in enumerate(chunks, 1):
                        count += 1
                        doc = {"source": p.name, "page": i+1, "pest": ", ".join(tags), "text": ch}
                        outp = f"{DATA_DIR}/pdf_doc_{count}.json"
                        with open(outp, "w", encoding="utf-8") as f:
                            json.dump(doc, f, ensure_ascii=False, indent=2)
            print("Processed", p.name)
        except Exception as e:
            print("Failed to process", p.name, ":", e)

print("Extraction complete. JSON docs saved to:", DATA_DIR)

Processed NRRI_Research-Bulletin-No-53.pdf
Extraction complete. JSON docs saved to: agri_corpus


In [25]:
corpus = []
doc_files = sorted(pathlib.Path(DATA_DIR).glob("*.json"))
for i, docf in enumerate(doc_files):
    d = json.load(open(docf, "r", encoding="utf-8"))
    corpus.append({"id": f"doc{i+1}", "text": d.get("text",""), "meta": {"source": d.get("source"), "page": d.get("page"), "pest": d.get("pest")}})

print("Corpus prepared. Chunks:", len(corpus))
# show a sample
if len(corpus) > 0:
    print(corpus[0]['meta'], corpus[0]['text'][:200])

Corpus prepared. Chunks: 103
{'source': 'NRRI_Research-Bulletin-No-53.pdf', 'page': 3, 'pest': 'brown planthopper'} NRRI Research BNuRRlIl eRetseianrc hN Boull.e-ti n5 030 BROWN PLANTHOPPER RESISTANT RICE: A JOURNEY FROM LANDRACES TO VARIETIES Guru Pirasanna Pandi G, Meera Kumari Kar, Mridul Chakraborti, Lambodar B


In [26]:
EMBED_MODEL = "all-MiniLM-L6-v2"
print("Loading embedder:", EMBED_MODEL)
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c['text'] for c in corpus]
if len(texts) == 0:
    raise SystemExit("No corpus texts found. Run the extraction step or add PDFs.")

# compute embeddings
embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True).astype('float32')
faiss.normalize_L2(embeddings)
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

# mapping
id_to_meta = {i: corpus[i] for i in range(len(corpus))}
print("FAISS index built. Vectors:", index.ntotal)

Loading embedder: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

FAISS index built. Vectors: 103


In [27]:
from typing import Tuple

def retrieve(query: str, k: int = 3) -> List[Dict[str, Any]]:
    q_emb = embedder.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = id_to_meta[int(idx)]
        results.append({"score": float(score), "text": meta['text'], "meta": meta['meta']})
    return results

print("Retriever ready. Try: retrieve('How to manage brown planthopper?', k=3)")

Retriever ready. Try: retrieve('How to manage brown planthopper?', k=3)


In [28]:
import re
from collections import OrderedDict

KEYWORDS = ["management","manage","control","monitor","avoid","apply","release","use resistant",
            "biological","neem","bt","trichogramma","pheromone","stubble","sanitation"]

def extract_management_sentences(retrieved: List[Dict[str,Any]]):
    sentences = []
    for r in retrieved:
        text = r['text']
        pieces = re.split(r'(?<=[\.\n])\s+', text)
        for s in pieces:
            s_clean = s.strip()
            if not s_clean:
                continue
            s_low = s_clean.lower()
            if any(kw in s_low for kw in KEYWORDS):
                s_clean = re.sub(r'\s+\.\.\..*$', '', s_clean)
                sentences.append((r['meta'], s_clean))
    return sentences


def build_answer_from_sentences(question: str, retrieved: List[Dict[str,Any]], max_actions: int = 6):
    candidates = extract_management_sentences(retrieved)
    if not candidates:
        return "No clear guidance in the provided evidence. See retrieved chunks for details.", [], []
    seen = OrderedDict()
    for meta, sent in candidates:
        if sent not in seen:
            seen[sent] = meta
    dedup = list(seen.items())
    summary_sents = [s for s,_ in dedup[:3]]
    summary = " ".join(summary_sents)
    actions = []
    for sent, meta in dedup:
        parts = re.split(r';|, and |, then | - |,', sent)
        for p in parts:
            p = p.strip()
            if len(p) > 12 and len(actions) < max_actions:
                p = p[0].upper() + p[1:]
                if p not in actions:
                    actions.append(p)
            if len(actions) >= max_actions:
                break
        if len(actions) >= max_actions:
            break
    if not actions:
        actions = summary_sents[:max_actions]
    evidence_used = []
    for i, r in enumerate(retrieved, 1):
        for sent, meta in dedup:
            if meta == r['meta'] and i not in evidence_used:
                evidence_used.append(i)
    return summary.strip(), actions, evidence_used

print("Deterministic answer builder ready.")

Deterministic answer builder ready.


In [37]:
q = "How to manage brown planthopper in rice?"
retr = retrieve(q, k=3)
print("Retrieved chunks:")
for i,r in enumerate(retr,1):
    print(i, round(r['score'],3), r['meta']['pest'], r['meta']['source'])

ans, actions, evidence_used = build_answer_from_sentences(q, retr, max_actions=6)
print('\nSHORT ANSWER:\n', ans)
print('\nRECOMMENDED ACTIONS:')
for i,a in enumerate(actions,1):
    print(i, a)
print('\nEvidence used:', evidence_used)

Retrieved chunks:
1 0.722 brown planthopper NRRI_Research-Bulletin-No-53.pdf
2 0.716 general ipm, brown planthopper NRRI_Research-Bulletin-No-53.pdf
3 0.706 brown planthopper NRRI_Research-Bulletin-No-53.pdf

SHORT ANSWER:
 Integrated pest management: concepts and approaches.

RECOMMENDED ACTIONS:
1 Integrated pest management: concepts and approaches.

Evidence used: [2]


In [30]:
app_code = r"""
import streamlit as st
import re
st.set_page_config(page_title='AgriAssist-RAG', layout='centered')
st.title('AgriAssist-RAG — Rice Pest Management (Deterministic)')
st.write('Domain-focused RAG: evidence-grounded answers for rice pests.')

question = st.text_input('Ask about rice pest management (e.g., How to manage brown planthopper?)')
k = st.sidebar.slider('Number of evidence chunks to retrieve (k)', 1, 5, 3)

if st.button('Get deterministic answer'):
    if not question.strip():
        st.warning('Please type a question first.')
    else:
        try:
            from __main__ import retrieve, build_answer_from_sentences
            retrieved = retrieve(question, k=k)
            summary, actions, evidence_used = build_answer_from_sentences(question, retrieved)
            st.markdown('### Short Answer')
            st.write(summary)
            st.markdown('### Recommended actions')
            for i,a in enumerate(actions,1):
                st.write(f"{i}. {a}")
            st.markdown('### Retrieved evidence (preview)')
            for i,r in enumerate(retrieved,1):
                st.write(f"[{i}] {r['meta']['source']} — {r['meta']['pest']} (score {r['score']:.3f})")
                st.write(r['text'][:300] + ('...' if len(r['text'])>300 else ''))
            st.caption(f"Evidence indices: {evidence_used}")
        except Exception as e:
            st.error('retrieve() or builder not available in this runtime. Run the notebook cells that build the index first.')
"""
with open('app.py','w',encoding='utf-8') as f:
    f.write(app_code)
print('Streamlit app written to app.py. Run with: streamlit run app.py')

Streamlit app written to app.py. Run with: streamlit run app.py


In [31]:
import pickle
np.save(f"{INDEX_DIR}/embeddings.npy", embeddings)
faiss.write_index(index, f"{INDEX_DIR}/faiss.index")
with open(f"{INDEX_DIR}/meta.pkl","wb") as f:
    pickle.dump(id_to_meta, f)
print('Saved index and metadata to', INDEX_DIR)

Saved index and metadata to index_data


In [32]:
print('\nNotebook complete. How to run:')
print('1) Run cells in order from top to bottom.')
print('2) After index built, test queries using the Demo Q&A cell (Cell 10).')
print('3) Run `streamlit run app.py` to launch the demo (if running locally or with a tunnel).')
print('\nIf a PDF link fails during download, remove or replace it in Cell 3 and re-run Cell 4.')


Notebook complete. How to run:
1) Run cells in order from top to bottom.
2) After index built, test queries using the Demo Q&A cell (Cell 10).
3) Run `streamlit run app.py` to launch the demo (if running locally or with a tunnel).

If a PDF link fails during download, remove or replace it in Cell 3 and re-run Cell 4.


In [33]:
# 1) Show downloaded pdfs
!ls -la download_pdfs || true

# 2) Show number of extracted json docs
!ls -la agri_corpus | sed -n '1,40p' || true

# 3) Check FAISS index vector count (run after Cell 7)
# (Only run after embeddings built)
try:
    print("FAISS vectors:", index.ntotal)
    print("Sample corpus size:", len(corpus))
except NameError:
    print("Index or corpus not available in this session yet.")


total 1576
drwxr-xr-x 2 root root    4096 Dec  9 16:24 .
drwxr-xr-x 1 root root    4096 Dec  9 16:52 ..
-rw-r--r-- 1 root root 1603666 Dec  9 16:24 NRRI_Research-Bulletin-No-53.pdf
total 420
drwxr-xr-x 2 root root 4096 Dec  9 16:48 .
drwxr-xr-x 1 root root 4096 Dec  9 16:52 ..
-rw-r--r-- 1 root root 1381 Dec  9 16:48 pdf_doc_100.json
-rw-r--r-- 1 root root 1281 Dec  9 16:48 pdf_doc_101.json
-rw-r--r-- 1 root root 1288 Dec  9 16:48 pdf_doc_102.json
-rw-r--r-- 1 root root  698 Dec  9 16:48 pdf_doc_103.json
-rw-r--r-- 1 root root  768 Dec  9 16:48 pdf_doc_10.json
-rw-r--r-- 1 root root 1267 Dec  9 16:48 pdf_doc_11.json
-rw-r--r-- 1 root root 1144 Dec  9 16:48 pdf_doc_12.json
-rw-r--r-- 1 root root  242 Dec  9 16:48 pdf_doc_13.json
-rw-r--r-- 1 root root 1597 Dec  9 16:48 pdf_doc_14.json
-rw-r--r-- 1 root root 1212 Dec  9 16:48 pdf_doc_15.json
-rw-r--r-- 1 root root  318 Dec  9 16:48 pdf_doc_16.json
-rw-r--r-- 1 root root 1350 Dec  9 16:48 pdf_doc_17.json
-rw-r--r-- 1 root root 1329 Dec  9

In [35]:
demo_questions = [
    "How to manage brown planthopper in rice?",
    "Cultural controls for stem borer?",
    "How to manage leaf folder?"
]
results = []
for q in demo_questions:
    retr = retrieve(q, k=3)
    ans, actions, evidence_used = build_answer_from_sentences(q, retr, max_actions=6)
    results.append({"question": q, "answer": ans, "actions": actions, "evidence_used": evidence_used, "retrieved": retr})
import json
with open("demo_results.json","w",encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Saved demo_results.json")


Saved demo_results.json


In [36]:
# example scoring function (keyword recall)
def score_answer(ans, keywords):
    a = ans.lower()
    hits = sum(1 for kw in keywords if kw.lower() in a)
    return hits / len(keywords)

# run across a test set, average scores, report mean recall
